In [ ]:
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Cambia 'NombreDeTuCarpeta' por el nombre real en tu Drive
base_path = '/content/drive/MyDrive/projects'
dataset_1 = os.path.join(base_path, 'recycling_project')

print(dataset_1)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define la ruta de tu carpeta principal en Drive
# Cambia 'Proyecto_Reciclaje' por el nombre real de tu carpeta
base_dir = '/content/drive/MyDrive/recycling_project'

# 2. Obtener los nombres de las clases (nombres de las subcarpetas)
clases = os.listdir(base_dir)

# 3. Contar cuántas imágenes hay en cada una
conteos = []
for clase in clases:
    ruta_clase = os.path.join(base_dir, clase)
    if os.path.isdir(ruta_clase):
        n_imagenes = len(os.listdir(ruta_clase))
        conteos.append(n_imagenes)
    else:
        # Esto ignora archivos sueltos como .DS_Store o .txt
        clases.remove(clase)

# 4. Crear el gráfico
plt.figure(figsize=(10, 6))
sns.barplot(x=conteos, y=clases, palette='rocket')
plt.title('Distribución de Imágenes por Clase', fontsize=15)
plt.xlabel('Tipo de Residuo', fontsize=12)
plt.ylabel('Cantidad de Imágenes', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Mostrar los números exactos encima de las barras
for i, v in enumerate(conteos):
    plt.text(v + 1, i, str(v), va='center', fontweight='bold')

plt.show()

### Arreglar colores de imágenes
1. Promedio de Intensidad por Canal (RGB)
Este método te permite ver si una clase de residuo es predominantemente más "roja", "verde" o "azul".

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

def eda_colores_pil(base_dir, sample_size=30):
    # Obtener las subcarpetas (clases)
    classes = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
    
    for class_name in classes:
        class_path = os.path.join(base_dir, class_name)
        # Filtrar solo archivos de imagen (jpg, png, jpeg)
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:sample_size]
        
        all_red, all_green, all_blue = [], [], []
        
        for img_name in image_files:
            img_path = os.path.join(class_path, img_name)
            try:
                # Abrir con PIL y asegurar que esté en RGB
                with Image.open(img_path) as img:
                    img = img.convert('RGB')
                    img = img.resize((100, 100)) # Reducimos para velocidad
                    img_array = np.array(img)
                    
                    # Extraer canales y guardarlos
                    all_red.extend(img_array[:, :, 0].flatten())
                    all_green.extend(img_array[:, :, 1].flatten())
                    all_blue.extend(img_array[:, :, 2].flatten())
            except Exception as e:
                print(f"No se pudo cargar {img_name}: {e}")

        # Graficar si encontramos datos
        if all_red:
            plt.figure(figsize=(10, 4))
            sns.kdeplot(all_red, color="red", label="Rojo", fill=True, alpha=0.1)
            sns.kdeplot(all_green, color="green", label="Verde", fill=True, alpha=0.1)
            sns.kdeplot(all_blue, color="blue", label="Azul", fill=True, alpha=0.1)
            
            plt.title(f'Distribución de Colores (PIL) - Clase: {class_name}', fontsize=14)
            plt.xlabel('Intensidad (0-255)')
            plt.ylabel('Densidad')
            plt.xlim(0, 255)
            plt.legend()
            plt.grid(axis='y', linestyle='--', alpha=0.3)
            plt.show()


eda_colores_pil(base_dir)
# Para ejecutarlo, descomenta la línea de abajo y pon tu ruta:
# eda_colores_pil('tu/ruta/al/dataset')
print()

In [ ]:
print("1. ¿POR QUÉ LAS MONTAÑAS SON TAN ALTAS? (DENSIDAD)")
print("--------------------------------------------------")
print("- La altura de la curva representa la frecuencia (cantidad de píxeles).")
print("- MONTAÑAS ALTAS: Miles de píxeles comparten el mismo tono. El pico en 255")
print("  es tu fondo blanco uniforme que 'ocupa' casi toda la imagen.")
print("- MONTAÑAS BAJAS: Indican colores que aparecen poco o son detalles mínimos.")
print("")

print("2. ¿POR QUÉ SE SEPARAN LOS COLORES? (RGB)")
print("-----------------------------------------")
print("- Las líneas son los canales: Rojo (R), Verde (G) y Azul (B).")
print("- LÍNEAS JUNTAS: El objeto es neutro (gris, blanco o negro) como la batería.")
print("- LÍNEAS SEPARADAS: Hay un color dominante.")
print("  * Brown-glass: El Rojo domina porque el marrón es derivado del rojo.")
print("  * Paper/Plastic: El Rojo y Verde dominan al Azul, indicando luz cálida.")
print("")

print("3. ¿QUÉ SIGNIFICAN LOS NÚMEROS? (INTENSIDAD 0-255)")
print("-------------------------------------------------")
print("- Es la escala de brillo que entiende la computadora:")
print("- 0 (Izquierda): NEGRO ABSOLUTO. Sombras y oscuridad total.")
print("- 127 (Centro): TONOS MEDIOS. Aquí está la textura real y el detalle.")
print("- 255 (Derecha): BLANCO PURO. Es el fondo o brillos que 'queman' la imagen.")
print("")

print("RESUMEN DE SALUD DEL DATASET")
print("----------------------------")
print("Tu dataset tiene un 'sesgo de fondo'. El modelo de IA podría estar")
print("aprendiendo a identificar fondos blancos en lugar de los objetos.")
print("Se recomienda normalizar o usar 'Data Augmentation' para equilibrar.")

In [ ]:
#(Análisis Exploratorio de Dimensiones) es crucial en visión artificial porque ayuda a decidir qué resolución de 
# entrada usar para tu modelo (como ResNet o MobileNet) y a detectar si hay imágenes corruptas o demasiado pequeñas.
# Para empezar, necesitas extraer el ancho (width) y el alto (height) de cada imagen. Aquí tienes una guía paso a paso usando PIL y Pandas:





In [ ]:
import os
import pandas as pd
from PIL import Image

def get_image_sizes(base_path):
    data = []
    classes = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
    
    for class_name in classes:
        class_path = os.path.join(base_path, class_name)
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_path, img_name)
                try:
                    with Image.open(img_path) as img:
                        width, height = img.size
                        data.append({
                            'clase': class_name,
                            'ancho': width,
                            'alto': height,
                            'aspect_ratio': width / height
                        })
                except:
                    continue
    return pd.DataFrame(data)

# Ejecución
df_sizes = get_image_sizes(base_dir)
print(df_sizes)


In [ ]:
# Agrupar por clase y ver estadísticas de ancho y alto
resumen_clases = df_sizes.groupby('clase')[['ancho', 'alto']].agg(['mean', 'min', 'max', 'count'])
print(resumen_clases)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
# Gráfico para el Ancho
plt.subplot(1, 2, 1)
sns.boxplot(data=df_sizes, x='clase', y='ancho', palette='viridis')
plt.xticks(rotation=45)
plt.title('Distribución de Ancho por Clase')

# Gráfico para el Alto
plt.subplot(1, 2, 2)
sns.boxplot(data=df_sizes, x='clase', y='alto', palette='rocket')
plt.xticks(rotation=45)
plt.title('Distribución de Alto por Clase')

plt.tight_layout()
plt.show()

In [ ]:
# Definimos umbrales basados en tus gráficos
umbral_bajo = 150  # Por debajo de esto son los "puntos" inferiores

print("📊 RESUMEN CRÍTICO DEL EDA DE TAMAÑO")
print("-" * 40)

# 1. Análisis de imágenes pequeñas (los puntos de abajo en tu gráfico)
imgs_pequenas = df_sizes[(df_sizes['ancho'] < umbral_bajo) | (df_sizes['alto'] < umbral_bajo)]
print(f"⚠️ Imágenes candidatas a eliminación (<{umbral_bajo}px): {len(imgs_pequenas)}")
print(f"   Clases más afectadas: \n{imgs_pequenas['clase'].value_counts().head(3)}")

print("-" * 40)

# 2. Promedios globales
print(f"📍 Resolución promedio: {int(df_sizes['ancho'].mean())}x{int(df_sizes['alto'].mean())}")
print(f"📏 Resolución máxima detectada: {df_sizes['ancho'].max()}x{df_sizes['alto'].max()}")

print("-" * 40)

# 3. Alerta de deformación
ratio_promedio = df_sizes['aspect_ratio'].mean()
print(f"🖼️ Relación de aspecto predominante: {ratio_promedio:.2f}")
if ratio_promedio > 1.2:
    print("🔔 NOTA: Tus imágenes son mayoritariamente horizontales.")
    print("   Si redimensionas a cuadrado (1:1) sin padding, los objetos se encogerán lateralmente.")

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(data=df_sizes, x='clase', y='aspect_ratio', inner="quartile")
plt.axhline(1.0, color='red', linestyle='--', label='Cuadrada (1:1)')
plt.title('Relación de Aspecto por Clase')
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Configuración de rutas
dir_principal = "/content/drive/MyDrive/recycling_project" 
# Solo obtenemos carpetas para evitar errores con archivos sueltos (.DS_Store, etc.)
clases = sorted([f for f in os.listdir(dir_principal) if os.path.isdir(os.path.join(dir_principal, f))])

# Crear el mapeo de nombres a IDs numéricos
mapeo_clases = {nombre: i for i, nombre in enumerate(clases)}
print("Mapeo generado:", mapeo_clases)

datos = []

# 2. Recolectar todas las rutas de imágenes y sus etiquetas
for clase in clases:
    ruta_clase = os.path.join(dir_principal, clase)
    for img in os.listdir(ruta_clase):
        if img.lower().endswith(('.png', '.jpg', '.jpeg')): # Filtro de extensiones
            datos.append({
                'file_path': os.path.join(ruta_clase, img),
                'label': clase,
                'label_id': mapeo_clases[clase] # Asignamos el ID según el mapeo
            })

# Convertir a DataFrame
df = pd.DataFrame(datos)

# 3. Crear los Splits (80% Train, 10% Val, 10% Test)
# Usamos stratify=df['label'] para que todas las clases tengan la misma proporción en los splits
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=1337, stratify=df['label'])

# Dividir el 20% restante en 10% Val y 10% Test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=1337, stratify=temp_df['label'])

# 4. Guardar los archivos CSV
train_df.to_csv('/content/drive/MyDrive/train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/val.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/test.csv', index=False)

print(f"✅ Procesamiento completado. Total de imágenes: {len(df)}")

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Cargar los archivos CSV desde tu Drive
train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/val.csv')

# Es vital que las etiquetas en el CSV sean strings para que el generador las reconozca
train_df['label'] = train_df['label'].astype(str)
val_df['label'] = val_df['label'].astype(str)

# 2. Configurar el Generador de Datos
# Nota: Usamos preprocess_input de MobileNetV2 en lugar de rescale=1/255
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# 3. Crear los cargadores de imágenes desde el DataFrame
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='file_path',  # La columna con la ruta completa
    y_col='label',      # La columna con el nombre de la clase
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='file_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# 4. Definir la Arquitectura
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    # Usamos el número de clases detectado por el generador
    layers.Dense(len(train_generator.class_indices), activation='softmax')])

# 5. Compilar
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6. Entrenar
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10 # Puedes subirlo a 20 o más
)